In [1]:
!pip install -q pandas matplotlib scikit-learn transformers datasets accelerate

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import wandb

from datasets import Dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments
from transformers import Trainer

from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay

d:\sentiment and sarcasm analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available. Training will be slower on CPU.")

CUDA available: True
GPU name: NVIDIA GeForce RTX 4050 Laptop GPU


In [4]:
os.makedirs("../outputs/tables", exist_ok=True)
os.makedirs("../outputs/figures", exist_ok=True)
os.makedirs("../outputs/predictions", exist_ok=True)
os.makedirs("../outputs/models", exist_ok=True)

In [5]:
sarcasm_df = pd.read_csv("../outputs/data/sarcasm.csv")

print("Data shape:", sarcasm_df.shape)
sarcasm_df.head()

Data shape: (12641, 9)


,text,label,variety,source,task,split,text_length,word_count,clean_text
0,"Located 2 blocks back from The Strand, ideal f...",0,en-AU,Google,Sarcasm,train,281,48,"located 2 blocks back from the strand, ideal f..."
1,Have n't been to AJ in a few years so popped i...,0,en-AU,Google,Sarcasm,train,368,75,have n't been to aj in a few years so popped i...
2,Tried their folded chili eggs() plus mushrooms...,0,en-AU,Google,Sarcasm,train,510,93,tried their folded chili eggs() plus mushrooms...
3,Thanks for the vegan options. Minus one star f...,0,en-AU,Google,Sarcasm,train,108,17,thanks for the vegan options. minus one star f...
4,Bought an ANGUS Bacon BBQ Sauce with Onion and...,0,en-AU,Google,Sarcasm,train,193,38,bought an angus bacon bbq sauce with onion and...


In [6]:
print("Label distribution:")
print(sarcasm_df["label"].value_counts().sort_index())

print("\nSplit distribution:")
print(sarcasm_df["split"].value_counts())

Label distribution:
label
0    10827
1     1814
Name: count, dtype: int64

Split distribution:
split
train         8894
test          2531
validation    1216
Name: count, dtype: int64


In [7]:
train_df = sarcasm_df[sarcasm_df["split"] == "train"].copy()
valid_df = sarcasm_df[sarcasm_df["split"] == "validation"].copy()
test_df = sarcasm_df[sarcasm_df["split"] == "test"].copy()

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

Train: (8894, 9)
Validation: (1216, 9)
Test: (2531, 9)


In [8]:
train_df = train_df[["text", "label"]]
valid_df = valid_df[["text", "label"]]
test_df = test_df[["text", "label", "variety", "source", "split"]]

print(train_df.head())

                                                text  label
0  Located 2 blocks back from The Strand, ideal f...      0
1  Have n't been to AJ in a few years so popped i...      0
2  Tried their folded chili eggs() plus mushrooms...      0
3  Thanks for the vegan options. Minus one star f...      0
4  Bought an ANGUS Bacon BBQ Sauce with Onion and...      0


In [9]:
train_data = Dataset.from_pandas(train_df)
valid_data = Dataset.from_pandas(valid_df)
test_data = Dataset.from_pandas(test_df)

print(train_data)
print(valid_data)
print(test_data)

Dataset({
    features: ['text', 'label'],
    num_rows: 8894
})
Dataset({
    features: ['text', 'label'],
    num_rows: 1216
})
Dataset({
    features: ['text', 'label', 'variety', 'source', 'split'],
    num_rows: 2531
})


In [10]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [11]:
max_length = 128

def tokenize_data(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=max_length
    )

In [12]:
train_data = train_data.map(tokenize_data, batched=True)
valid_data = valid_data.map(tokenize_data, batched=True)
test_data = test_data.map(tokenize_data, batched=True)

print(train_data[0].keys())

Map: 100%|██████████| 2531/2531 [00:00<00:00, 17763.70 examples/s]

dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])


In [13]:
num_labels = sarcasm_df["label"].nunique()

print("Number of labels:", num_labels)
print("Labels:", sorted(sarcasm_df["label"].unique()))

Number of labels: 2
Labels: [np.int64(0), np.int64(1)]


In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5856.65it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [15]:
def compute_metrics(eval_result):
    logits, labels = eval_result
    predictions = np.argmax(logits, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro")
    macro_precision = precision_score(labels, predictions, average="macro", zero_division=0)
    macro_recall = recall_score(labels, predictions, average="macro", zero_division=0)
    
    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall
    }

In [16]:
wandb.init(
    project="sentiment-sarcasm-besstie",
    name="bert-base-sarcasm",
    config={
        "model": "bert-base-uncased",
        "task": "Sarcasm",
        "max_length": 128,
        "learning_rate": 2e-5,
        "batch_size": 8,
        "gradient_accumulation_steps": 2,
        "epochs": 3,
        "weight_decay": 0.01
    }
)

NameError: name 'wandb' is not defined